In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor
import os

from vla.constants import PROJECT_ROOT

# Your custom imports
from vla.models.smolvla import SmolVLAPolicy

In [ ]:
checkpoint = "HuggingFaceVLA/smolvla_libero"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model from checkpoint: {checkpoint} on device: {device}")
model = SmolVLAPolicy(checkpoint, action_dim=7, device=device)
hf_vlm = model.model.vlm_with_expert.vlm
llm = hf_vlm.model.text_model
total_layers = len(llm.layers)
print(f"Isolated the Language Model successfully. Total layers: {total_layers}")

In [ ]:
# Choose suite
suite_name = "object"
task_description = os.listdir(PROJECT_ROOT / "data/images/libero" / suite_name)[0]

# Load Full Task Description
task_path = (
    PROJECT_ROOT / "data/images/libero" / suite_name / task_description / "task.txt"
)

with open(task_path) as f:
    task_description = f.read().strip()

print(f"Task: {task_description}")

In [ ]:
target_word = "juice" # Pick a specific word to track (e.g., "bowl" or "box")
PROBE_LAYER = 19  # Layer shown to focus on objects (from vlm_diagnostic.ipynb)

## Step 1 — Extract hidden states for the "juice" token across all frames

For each frame in the episode, we run a forward pass through the VLM and extract the hidden state
of the `target_word` token at a chosen LLM layer. This gives us a matrix of shape
`(num_frames, hidden_dim)` which we can use as features for the linear probe.


In [ ]:

import numpy as np
from pathlib import Path

frame_dir = PROJECT_ROOT / "data/images/libero/object/ep0000_task_0"
frame_paths = sorted(frame_dir.glob("frame*.png"))
print(f"Found {len(frame_paths)} frames")

# --- Processor setup ---
try:
    processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
except Exception:
    processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

llm = hf_vlm.model.text_model
vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype


image_special_ids: set[int] = set()
for tok in processor.tokenizer.additional_special_tokens:
    if any(kw in tok.lower() for kw in ["image", "img", "row", "col", "fake_token", "global"]):
        image_special_ids.add(processor.tokenizer.convert_tokens_to_ids(tok))
# Always include the primary image placeholder token
image_special_ids.add(processor.tokenizer.convert_tokens_to_ids(processor.image_token))
print(f"Image-related special token IDs filtered out: {image_special_ids}")

# --- Build prompt ---
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task_description}]}]
prompt_str = processor.apply_chat_template(messages, add_generation_prompt=True)


def extract_token_hidden_state(pil_image, layer_idx, target_word):
    """
    Returns the hidden state vector for the FIRST occurrence of target_word
    in the token sequence, at the given LLM layer. Shape: (hidden_dim,)

    SmolVLM2 compresses image tokens via pixel-shuffle before the LLM.
    The vision encoder always produces N_IMAGE_TOKENS_LLM=64 tokens in the
    LLM hidden-state sequence. We use this fixed constant to remap the text
    token index, and filter ALL image-related special tokens (not just the
    main image placeholder) when computing the text-token offset.
    """
    captured = {}

    def hook(module, input, output):
        # output[0] may be (seq, hidden) or (B, seq, hidden) - output 0 = right for hidden states, output 1 = attentions
        h = output[0].detach().cpu().float()
        captured["hidden"] = h

    handle = llm.layers[layer_idx].register_forward_hook(hook)

    inputs = processor(text=prompt_str, images=pil_image, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    with torch.no_grad():
        hf_vlm(**inputs)

    handle.remove()

    hidden = captured["hidden"]
    # Normalise to (seq, hidden)
    if hidden.dim() == 3:
        hidden = hidden[0]  # (B, seq, hidden) → (seq, hidden)
    hidden_seq_len = hidden.shape[0]

    # --- Remap target token index to compressed LLM sequence ---
    input_ids_list = inputs["input_ids"][0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(input_ids_list)

    # Filter ALL image-related special tokens so only true text tokens remain.
    text_positions = [i for i, tid in enumerate(input_ids_list) if tid not in image_special_ids]

    # Find target word among text tokens
    target_text_offset = next(
        (j for j, i in enumerate(text_positions) if target_word.lower() in tokens[i].lower()), None
    )

    if target_text_offset is None:
        raise ValueError(f"Token '{target_word}' not found in text tokens: "
                         f"{[tokens[i] for i in text_positions]}")

    target_idx = next(
    i for i, tok in enumerate(tokens)
    if target_word.lower() in tok.lower().replace("▁", "")
    )
    print(f"  '{target_word}' → text offset {target_text_offset}, "
          f"LLM idx {target_idx} / {hidden_seq_len}")

    return hidden[target_idx, :].numpy()  # (hidden_dim,)


#Extract for all frames 
hidden_states = []
for i, path in enumerate(frame_paths):
    img = Image.open(path).convert("RGB")
    h = extract_token_hidden_state(img, PROBE_LAYER, target_word)
    hidden_states.append(h)
    if (i + 1) % 20 == 0:
        print(f"  Processed {i + 1}/{len(frame_paths)} frames")

X = np.stack(hidden_states)  # (num_frames, hidden_dim)
frame_indices = np.arange(len(frame_paths))   # label: task progress (0 → end)
print(f"\nFeature matrix shape: {X.shape}")


## Step 2 — PCA: Does the representation drift over the episode?

If the hidden state of `"juice"` geometrically changes as the task progresses,
this 2D PCA scatter (colored by frame index) should show a gradient or trajectory —
evidence that the model encodes task-progress state in the token representation.


In [ ]:

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=frame_indices, cmap="plasma", s=20)
plt.colorbar(sc, ax=ax, label="Frame index (task progress)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title(f'PCA of "{target_word}" hidden state @ layer {PROBE_LAYER}')
plt.tight_layout()
plt.show()

print(f"Variance explained by PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")


## Step 3 — Regression probe: Can a linear model predict task progress?

We predict not, but lets see...

We train a `Ridge` regression on 80% of frames and evaluate on the remaining 20%.
- **High $R^2$** → the hidden state linearly encodes where we are in the task.
- **Low $R^2$** → task progress is either not encoded, or encoded non-linearly.

In [ ]:

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# --- Single-layer probe at PROBE_LAYER ---
y = frame_indices.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, shuffle=False)

probe = Ridge(alpha=1.0)
probe.fit(X_tr, y_tr)
y_pred = probe.predict(X_te)
r2 = r2_score(y_te, y_pred)
print(f"Layer {PROBE_LAYER} regression probe  R² = {r2:.3f}")

# --- Plot predictions vs ground truth ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(y_te, label="True frame index", linewidth=1.5)
ax.plot(y_pred, label="Predicted frame index", linewidth=1.5, linestyle="--")
ax.set_xlabel("Test sample")
ax.set_ylabel("Frame index")
ax.set_title(f'Linear probe prediction @ layer {PROBE_LAYER}  (R²={r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()


So it is not linear, lets try MLP

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

y = frame_indices.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, shuffle=False)

probe = MLPRegressor(
    hidden_layer_sizes=(128, 64),  # two hidden layers
    activation="relu",
    max_iter=1000,
    random_state=42
)
probe.fit(X_tr, y_tr)
y_pred = probe.predict(X_te)
r2 = r2_score(y_te, y_pred)
print(f"Layer {PROBE_LAYER} MLP probe  R² = {r2:.3f}")

# --- Plot predictions vs ground truth ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(y_te, label="True frame index", linewidth=1.5)
ax.plot(y_pred, label="Predicted frame index", linewidth=1.5, linestyle="--")
ax.set_xlabel("Test sample")
ax.set_ylabel("Frame index")
ax.set_title(f'Linear probe prediction @ layer {PROBE_LAYER}  (R²={r2:.3f})')
ax.legend()
plt.tight_layout()
plt.show()


Not very good, oh well

could be because of the output it needs. Maybe a classifier could be better given class1 = "not picked up", class2 = "picked up object" bla bla, well see if we continue this probing

## Step 4 — Layer sweep: Which layer best encodes task progress?

Re-extract features at every layer and score the regression probe.
This shows the full depth profile of where task-progress information lives.


In [ ]:
# --- All-layers-at-once extraction ---

def extract_all_layers_hidden_state(pil_image, target_word):
    """
    Single forward pass through the VLM.
    Returns a dict  {layer_idx: np.ndarray of shape (hidden_dim,)}
    for every LLM layer.
    """
    captured = {}
    handles = []

    for layer_idx, layer in enumerate(llm.layers):
        def make_hook(idx):
            def hook(module, input, output):
                h = output[0].detach().cpu().float()
                if h.dim() == 3:
                    h = h[0]
                captured[idx] = h
            return hook
        handles.append(layer.register_forward_hook(make_hook(layer_idx)))

    inputs = processor(text=prompt_str, images=pil_image, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    with torch.no_grad():
        hf_vlm(**inputs)

    for h in handles:
        h.remove()

    # Resolve the target token index once (same for every layer)
    input_ids_list = inputs["input_ids"][0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(input_ids_list)
    target_idx = next(
        i for i, tok in enumerate(tokens)
        if target_word.lower() in tok.lower().replace("▁", "")
    )

    return {idx: captured[idx][target_idx, :].numpy() for idx in captured}


# --- One pass per frame, store activations for all layers to avoid a verryyyyy long computation ---
print(f"Extracting all-layer activations for {len(frame_paths)} frames "
      f"({total_layers} layers each) …")

all_layer_feats = {idx: [] for idx in range(total_layers)}

for i, path in enumerate(frame_paths):
    img = Image.open(path).convert("RGB")
    layer_vecs = extract_all_layers_hidden_state(img, target_word)
    for idx, vec in layer_vecs.items():
        all_layer_feats[idx].append(vec)
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(frame_paths)} frames done")

print("Extraction complete. Running probes …")

# calc R^2
layer_r2_scores = {}

for layer_idx in range(total_layers):
    Xl = StandardScaler().fit_transform(np.stack(all_layer_feats[layer_idx]))
    Xl_tr, Xl_te, y_tr, y_te = train_test_split(Xl, y, test_size=0.2, shuffle=False)
    probe_l = Ridge(alpha=1.0).fit(Xl_tr, y_tr)
    layer_r2_scores[layer_idx] = r2_score(y_te, probe_l.predict(Xl_te))
    print(f"  Layer {layer_idx:2d}  R² = {layer_r2_scores[layer_idx]:.3f}")

# Plot
layers = list(layer_r2_scores.keys())
scores = [layer_r2_scores[l] for l in layers]

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["tab:orange" if l == PROBE_LAYER else "tab:blue" for l in layers]
ax.bar(layers, scores, color=colors)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Layer index")
ax.set_ylabel("R² (task-progress regression)")
ax.set_title(f'Linear probe R² per layer  (target token: "{target_word}")\nOrange = layer {PROBE_LAYER} used in vlm_diagnostic')
plt.tight_layout()
plt.show()


Hmmm all is negative for RIdge doubt MLP will do better

Final step of probing, needs implementation but here is a desciption:

## Step 5 — Binary classification probe: "juice grasped" vs. "juice not grasped"

We split the episode at the midpoint as a rough proxy:
- **Label 0** (first half): juice is on the table, not yet picked up.
- **Label 1** (second half): juice is being moved / placed.

A logistic regression accuracy well above 50% means the hidden state carries information
about the object's manipulation state — even though the model was never trained to classify this.


We should find the official image were the model has managed to grasp the juice and then train a model on this

This is kinda what we talked about in step3 

## Interpreting results

| Result | Interpretation |
|---|---|
| PCA shows a gradient from frame 0 → 142 | Representation drifts with task progress — model encodes state changes |
| Regression R² ≈ 1.0 | Hidden state is almost perfectly predictive of where in the task we are |
| Regression R² ≈ 0.0 | No linear task-progress information at this layer |
| Binary acc ≈ 100% | Model clearly separates "juice on table" from "juice moving/placed" |
| Binary acc ≈ 50% | No manipulation-state information linearly decodable |

The key takeaway compared to `.generate()` output: **even if text generation hallucinates,
the hidden states may still faithfully encode the true scene state**. Probing tests this directly.
